In [ ]:
import torch
from PIL import Image
import os
import numpy as np
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import yaml
from cleanfid import fid
import lpips
import torchvision.transforms as T
import itertools
import random
import collections
import re

with open("/work/cvcs2026/stochastic_parrots/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Definition Output Directories

In [ ]:
REF_IMAGES_DIR = config["paths"]["instance_images_dir"]
EVALUATION_DIR = config["paths"]["evaluation_dir"]
PROMPTS_DIR    = config["paths"]["prompts_dir"]

OUTPUT_CLIPI_SDXL = EVALUATION_DIR + "/metrics/sdxl/clipi"
OUTPUT_CLIPI_TI   = EVALUATION_DIR + "/metrics/ti-v1/clipi"

OUTPUT_CLIPT_SDXL = EVALUATION_DIR + "/metrics/sdxl/clipt"
OUTPUT_CLIPT_TI   = EVALUATION_DIR + "/metrics/ti-v1/clipt"

OUTPUT_FID_SDXL = EVALUATION_DIR + "/metrics/sdxl/fid"
OUTPUT_FID_TI   = EVALUATION_DIR + "/metrics/ti-v1/fid"

OUTPUT_LPIPS_SDXL = EVALUATION_DIR + "/metrics/sdxl/lpips"
OUTPUT_LPIPS_TI   = EVALUATION_DIR + "/metrics/ti-v1/lpips"

model_id = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_id).to("cuda")
processor = CLIPProcessor.from_pretrained(model_id)

# CLIP-I Metric
Test for **Personalization Quality**. How well does TI capture the target concept?

Computes cosine similarity between generated images and the reference instance images.

## Helper Functions

In [ ]:
def get_features(image_paths):
    images = [Image.open(p).convert("RGB") for p in image_paths]
    inputs = processor(images=images, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
        if torch.is_tensor(outputs):
            features = outputs
        else:
            features = getattr(outputs, "image_embeds", outputs[0])
        if len(features.shape) == 3:
            features = features[:, 0, :]
        features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features


ref_paths = [os.path.join(REF_IMAGES_DIR, f)
             for f in os.listdir(REF_IMAGES_DIR) if f.endswith(('.png', '.jpg'))]
ref_features = get_features(ref_paths)
avg_ref_feature = ref_features.mean(dim=0, keepdim=True)
avg_ref_feature /= avg_ref_feature.norm(p=2, dim=-1, keepdim=True)


def evaluate_folder(folder_path):
    gen_paths = [os.path.join(folder_path, f)
                 for f in os.listdir(folder_path) if f.endswith('.png')]
    if not gen_paths:
        return 0, 0
    all_similarities = []
    for path in gen_paths:
        gen_feat = get_features([path])
        sim = (gen_feat @ avg_ref_feature.T).item()
        all_similarities.append(sim)
    return np.mean(all_similarities), np.std(all_similarities)

## Execution

In [ ]:
mean_base, std_base = evaluate_folder(OUTPUT_CLIPI_SDXL)
mean_ti,   std_ti   = evaluate_folder(OUTPUT_CLIPI_TI)

print("\n--- RESULTS: SUBJECT FIDELITY (CLIP-I) ---")
print(f"{'Model':<25} | {'Mean':<15} | {'Std':<15}")
print(f"{'SDXL Base':<25} | {mean_base:<15.4f} | {std_base:<15.4f}")
print(f"{'TI v1':<25} | {mean_ti:<15.4f} | {std_ti:<15.4f}")
print(f"Fidelity increment: {((mean_ti - mean_base) / mean_base) * 100:.2f}%")

# CLIP-T Metric
Test for **Language Drift**. Does TI training cause the model to forget unrelated concepts?

Computes cosine similarity between 25 non-cat prompts and their generated images.

## Helper Functions

In [ ]:
def load_prompts(metric, model_name):
    base_prompt_dir = os.path.join(PROMPTS_DIR, metric)
    model_dir = os.path.join(base_prompt_dir, model_name)
    prompt_file = os.path.join(model_dir, "prompt.txt") if os.path.isdir(model_dir) \
                  else os.path.join(base_prompt_dir, "prompt.txt")
    print(f"Loading prompts from: {prompt_file}")
    with open(prompt_file, "r") as f:
        return [line.strip() for line in f if line.strip()]

prompts_clipt = load_prompts("clipt", "sdxl")

In [ ]:
def compute_clip_t(image_path, text_prompt):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(text=[text_prompt], images=[img],
                       return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        outputs = model(**inputs)
        image_embeds = outputs.image_embeds
        text_embeds  = outputs.text_embeds
        image_embeds /= image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds  /= text_embeds.norm(p=2, dim=-1, keepdim=True)
        return (text_embeds @ image_embeds.T).item()


def evaluate_clipt(folder_path, nprompt):
    p = prompts_clipt[nprompt]
    n = str(nprompt + 1).zfill(2)
    files = [os.path.join(folder_path, f)
             for f in os.listdir(folder_path) if f.startswith(f"image_{n}")]
    scores = [compute_clip_t(f, p) for f in files]
    return np.mean(scores), np.std(scores)

## Execution

In [ ]:
results = []
print(f"{'Prompt':<25} | {'Base':<15} | {'TI v1':<15} | {'Drift (%)':<10}")
print("-" * 75)
for i, prompt in enumerate(prompts_clipt):
    mean_base, std_base = evaluate_clipt(OUTPUT_CLIPT_SDXL, i)
    mean_ti,   std_ti   = evaluate_clipt(OUTPUT_CLIPT_TI,   i)
    drift = (1 - (mean_ti / mean_base)) * 100 if mean_base != 0 else 0
    results.append({"prompt": prompt, "base": mean_base, "ti": mean_ti, "drift": drift})
    print(f"{prompt[:25]:<25} | {mean_base:.4f} ± {std_base:.4f} "
          f"| {mean_ti:.4f} ± {std_ti:.4f} | {drift:.2f}%")

avg_base = np.mean([r["base"] for r in results])
avg_ti   = np.mean([r["ti"]   for r in results])
avg_drift = (1 - (avg_ti / avg_base)) * 100
print("-" * 75)
print(f"Average CLIP-T Base: {avg_base:.4f} | Average CLIP-T TI: {avg_ti:.4f} | Avg Drift: {avg_drift:.2f}%")

# FID Metric
Test for **Image Quality**. How different is the distribution of TI-generated images from the SDXL base?

Lower FID = more similar distributions (better quality preservation).

## Execution

In [ ]:
score_fid = fid.compute_fid(OUTPUT_FID_SDXL, OUTPUT_FID_TI, device=torch.device("cuda"))
print(f"FID Score (SDXL vs TI-v1): {score_fid:.4f}")

# LPIPS Metric
Test for **Mode Collapse**. Does TI generate diverse images or always the same output?

Higher LPIPS = more diverse images within the same prompt (better).

## Helper Functions

In [ ]:
loss_fn_alex = lpips.LPIPS(net='alex').to("cuda")

def load_and_preprocess(path):
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    return transform(img).unsqueeze(0).to("cuda")


def compute_internal_diversity(folder_path, max_pairs_per_prompt=2000):
    files = [os.path.join(folder_path, f)
             for f in os.listdir(folder_path) if f.endswith('.png')]
    if not files:
        return 0

    groups = collections.defaultdict(list)
    for f in files:
        match = re.search(r'image_(\d+)_', f)
        prompt_idx = match.group(1) if match else "unknown"
        groups[prompt_idx].append(f)

    all_prompt_distances = {}
    for prompt_idx, group_files in groups.items():
        if len(group_files) < 2:
            continue
        all_pairs = list(itertools.combinations(group_files, 2))
        sampled_pairs = random.sample(all_pairs, min(len(all_pairs), max_pairs_per_prompt))

        unique_files = {f for pair in sampled_pairs for f in pair}
        img_cache = {f: load_and_preprocess(f) for f in unique_files}

        distances = []
        for path_a, path_b in sampled_pairs:
            with torch.no_grad():
                dist = loss_fn_alex(img_cache[path_a], img_cache[path_b])
                distances.append(dist.item())

        avg_dist = sum(distances) / len(distances)
        all_prompt_distances[prompt_idx] = avg_dist
        print(f"  -> Average LPIPS for prompt {prompt_idx}: {avg_dist:.4f}")

        del img_cache
        torch.cuda.empty_cache()

    if not all_prompt_distances:
        return 0
    return sum(all_prompt_distances.values()) / len(all_prompt_distances)

## Execution

In [ ]:
print("--- SDXL Base ---")
div_base = compute_internal_diversity(OUTPUT_LPIPS_SDXL)

print("\n--- TI v1 ---")
div_ti = compute_internal_diversity(OUTPUT_LPIPS_TI)

print("-" * 30)
print(f"Internal Diversity (LPIPS) SDXL Base: {div_base:.4f}")
print(f"Internal Diversity (LPIPS) TI v1:     {div_ti:.4f}")

# Summary — SDXL Base vs TI v1

Raccolta di tutti i risultati in una tabella unica.

In [ ]:
print("="*60)
print(f"{'Metric':<20} | {'SDXL Base':<15} | {'TI v1':<15}")
print("="*60)
print(f"{'CLIP-I (↑)':<20} | {mean_base:<15.4f} | {mean_ti:<15.4f}")
print(f"{'CLIP-T (↑)':<20} | {avg_base:<15.4f} | {avg_ti:<15.4f}")
print(f"{'FID (↓)':<20} | {'(baseline)':<15} | {score_fid:<15.4f}")
print(f"{'LPIPS (↑)':<20} | {div_base:<15.4f} | {div_ti:<15.4f}")
print("="*60)
print("↑ = higher is better | ↓ = lower is better")